In [1]:
import os
# Avoid HDF5 file-locking errors when writing directly to an sshfs/FUSE mount (hpc_mount)
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")

from config import config
import src.db_read as dbrd
import src.save_grid as svgrd
import src.grid_topol as grdtpl
import src.weather as wth
import src.parallel_export as pexp

import pandas as pd
import numpy as np
import geopandas as gpd
from pyproj import Transformer

### Load valid grids (no sampling)
All filtered grids from `input_data/valid_grids` are exported to `.h5` without census-cell sampling.

In [2]:
df_grid_set = pd.read_hdf("input_data/valid_grids")  # Source: pylovo

# Keep x_census/y_census for 1 km weather cell grouping (stripped before saving /raw_data/region)
df_grids = (
    df_grid_set.rename(columns={"plz": "plz_pylovo", "Einwohner": "pop_density_1km2_cell"})
    .reset_index(drop=True)
    .copy()
)
print(f"Loaded {len(df_grids)} valid grids (no sampling).")
df_grids.head()

Loaded 85613 valid grids (no sampling).


,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,x_census,y_census
0,82467,3,13,4.404823e+06,2.709490e+06,4613.0,4404500.0,2709500.0
1,82467,3,14,4.405105e+06,2.708002e+06,1563.0,4405500.0,2708500.0
2,82467,3,15,4.405181e+06,2.708633e+06,1563.0,4405500.0,2708500.0
3,86941,1,1,4.398715e+06,2.776425e+06,115.0,4398500.0,2776500.0
4,82467,3,16,4.405167e+06,2.708841e+06,1563.0,4405500.0,2708500.0


In [3]:
print("Duplicate rows:", df_grids.duplicated().sum())
df_grids["pop_density_1km2_cell"].describe()

Duplicate rows: 0


count    85613.000000
mean      1506.848166
std       2599.136819
min          3.000000
25%        102.000000
50%        534.000000
75%       1720.000000
max      22773.000000
Name: pop_density_1km2_cell, dtype: float64

#### Assign zip-code and RegioStaR7

Latitude and Longitude

In [4]:
# Vectorized transformation from Zensus LAEA (EPSG:3035) to WGS84 (EPSG:4326)
transformer = Transformer.from_crs("EPSG:3035", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(df_grids["x"].to_numpy(), df_grids["y"].to_numpy())
census_lon, census_lat = transformer.transform(
    df_grids["x_census"].to_numpy(), df_grids["y_census"].to_numpy()
)
df_grids["lon"] = lon
df_grids["lat"] = lat
df_grids["census_lon"] = census_lon
df_grids["census_lat"] = census_lat
df_grids.head()

,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,x_census,y_census,lon,lat,census_lon,census_lat
0,82467,3,13,4.404823e+06,2.709490e+06,4613.0,4404500.0,2709500.0,11.111686,47.493358,11.107405,47.493492
1,82467,3,14,4.405105e+06,2.708002e+06,1563.0,4405500.0,2708500.0,11.115133,47.479931,11.120471,47.484357
2,82467,3,15,4.405181e+06,2.708633e+06,1563.0,4405500.0,2708500.0,11.116268,47.485601,11.120471,47.484357
3,86941,1,1,4.398715e+06,2.776425e+06,115.0,4398500.0,2776500.0,11.042857,48.096449,11.039989,48.097154
4,82467,3,16,4.405167e+06,2.708841e+06,1563.0,4405500.0,2708500.0,11.116126,47.487468,11.120471,47.484357


Zip code

In [5]:
shapefile_path = "input_data/plz-5stellig.shp"  # Source: https://www.suche-postleitzahl.org/downloads
gdf_zip = gpd.read_file(shapefile_path)

# Ensure the shapefile CRS is WGS84 (EPSG:4326) for spatial join
if gdf_zip.crs is None:
    gdf_zip = gdf_zip.set_crs(epsg=4326)
elif gdf_zip.crs.to_epsg() != 4326:
    gdf_zip = gdf_zip.to_crs(epsg=4326)

In [6]:
# Build GeoDataFrame of grid points once for fast vectorized spatial joins
gdf_points = gpd.GeoDataFrame(
    {"_orig_idx": df_grids.index},
    geometry=gpd.points_from_xy(df_grids["lon"], df_grids["lat"]),
    index=df_grids.index,
    crs="EPSG:4326",
)

# Vectorized point-in-polygon lookup (keeps first match per grid if polygons overlap)
joined_zip = gpd.sjoin(gdf_points, gdf_zip[["plz", "geometry"]], how="left", predicate="within")
joined_zip = joined_zip.drop_duplicates(subset=["_orig_idx"], keep="first").set_index("_orig_idx")
df_grids["plz"] = joined_zip["plz"]

df_grids = df_grids.dropna(subset=["plz"])  # Drop points outside German PLZ polygons
df_grids.head()

,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,x_census,y_census,lon,lat,census_lon,census_lat,plz
0,82467,3,13,4.404823e+06,2.709490e+06,4613.0,4404500.0,2709500.0,11.111686,47.493358,11.107405,47.493492,82467
1,82467,3,14,4.405105e+06,2.708002e+06,1563.0,4405500.0,2708500.0,11.115133,47.479931,11.120471,47.484357,82467
2,82467,3,15,4.405181e+06,2.708633e+06,1563.0,4405500.0,2708500.0,11.116268,47.485601,11.120471,47.484357,82467
3,86941,1,1,4.398715e+06,2.776425e+06,115.0,4398500.0,2776500.0,11.042857,48.096449,11.039989,48.097154,86941
4,82467,3,16,4.405167e+06,2.708841e+06,1563.0,4405500.0,2708500.0,11.116126,47.487468,11.120471,47.484357,82467


Gemeindeschlüssel

In [7]:
# Path to your downloaded VG250 shapefile:
MUNICI_SHP = "input_data/VG250_GEM.shp"  # Source: https://gdz.bkg.bund.de/

gdf_munic = gpd.read_file(MUNICI_SHP)
if gdf_munic.crs.to_epsg() != 4326:
    gdf_munic = gdf_munic.to_crs(epsg=4326)

In [8]:
# Subset gdf_points to remaining rows after PLZ filtering and join municipality AGS + name
gdf_points_valid = gdf_points.loc[df_grids.index]
joined_munic = gpd.sjoin(gdf_points_valid, gdf_munic[["AGS", "GEN", "geometry"]], how="left", predicate="within")
joined_munic = joined_munic.drop_duplicates(subset=["_orig_idx"], keep="first").set_index("_orig_idx")

df_grids["gemeindeschlüssel"] = joined_munic["AGS"]
df_grids["name"] = joined_munic["GEN"]

df_grids = df_grids.dropna(subset=["gemeindeschlüssel"])  # Drop points without matching municipality
df_grids.head()

,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,x_census,y_census,lon,lat,census_lon,census_lat,plz,gemeindeschlüssel,name
0,82467,3,13,4.404823e+06,2.709490e+06,4613.0,4404500.0,2709500.0,11.111686,47.493358,11.107405,47.493492,82467,09180117,Garmisch-Partenkirchen
1,82467,3,14,4.405105e+06,2.708002e+06,1563.0,4405500.0,2708500.0,11.115133,47.479931,11.120471,47.484357,82467,09180117,Garmisch-Partenkirchen
2,82467,3,15,4.405181e+06,2.708633e+06,1563.0,4405500.0,2708500.0,11.116268,47.485601,11.120471,47.484357,82467,09180117,Garmisch-Partenkirchen
3,86941,1,1,4.398715e+06,2.776425e+06,115.0,4398500.0,2776500.0,11.042857,48.096449,11.039989,48.097154,86941,09181118,Eresing
4,82467,3,16,4.405167e+06,2.708841e+06,1563.0,4405500.0,2708500.0,11.116126,47.487468,11.120471,47.484357,82467,09180117,Garmisch-Partenkirchen


RegioStaR7

In [9]:
df_regiostar = pd.read_excel("input_data/regiostar-referenzdateien.xlsx", sheet_name="ReferenzGebietsstand2020")
df_regiostar.head()

,gem_20,gemrs_20,name_20,bev_20,fl_20,vbgem_20,vbgemrs_20,vbgnam_20,land_20,RegioStaR2,RegioStaR4,RegioStaR17,RegioStaR7,RegioStaR5,RegioStaRGem7,RegioStaRGem5,RegioStaR_Stadtregion,RegioStaR_NameStadtregion
0,1001000,10010000000,"Flensburg, Stadt",89934,56.73,1001000,10010000,"Flensburg, Stadt",1,2,22,221,75,54,74,53,NaN,NaN
1,1002000,10020000000,"Kiel, Landeshauptstadt",246601,118.65,1002000,10020000,"Kiel, Landeshauptstadt",1,1,12,121,72,52,72,52,1002000.0,Kiel
2,1003000,10030000000,"Lübeck, Hansestadt",215846,214.19,1003000,10030000,"Lübeck, Hansestadt",1,1,12,121,72,52,72,52,1003000.0,Lübeck
3,1004000,10040000000,"Neumünster, Stadt",79905,71.66,1004000,10040000,"Neumünster, Stadt",1,2,21,211,75,54,74,53,NaN,NaN
4,1051001,10515175001,Albersdorf,3714,17.12,1051975,10515175,Mitteldithmarschen,1,2,22,225,77,55,77,55,NaN,NaN


In [10]:
# Map Gemeindeschlüssel (AGS) -> RegioStaR7 directly
ags_to_regio7 = df_regiostar.drop_duplicates(subset=["gem_20"]).set_index("gem_20")["RegioStaR7"]
df_grids["regio7"] = pd.to_numeric(df_grids["gemeindeschlüssel"], errors="coerce").map(ags_to_regio7)
df_grids = df_grids.dropna(subset=["regio7"])  # Drop grids without RegioStaR7 match

In [11]:
# Drop any duplicate rows so each grid is exported only once
df_grids = df_grids.drop_duplicates().reset_index(drop=True)

#### Assign DSO short name and deterministic `grid_index`

In [12]:
# Mapping from full DSO name in Zuordnung_PLZ-VNB.xlsx to filename short code
DSO_SHORT_NAMES = {
    "AllgäuNetz GmbH & Co. KG": "AllgN",
    "Bayernwerk Netz GmbH": "ByWrk",
    "Mainfranken Netze GmbH": "MFrak",
    "N-ERGIE Netz GmbH": "Nrgie",
    "nVNB": "nVNB",
    "Regensburg Netz GmbH": "RegB",
    "Stadtwerke Ulm/Neu-Ulm Netze GmbH": "SWU",
    "swa Netze GmbH": "SWA",
    "SWM Infrastruktur GmbH & Co. KG": "SWM",
    "LEW Verteilnetz GmbH": "LEW",
}

df_vnb = pd.read_excel("input_data/Zuordnung_PLZ-VNB.xlsx", sheet_name="plz_vnb_bayern")
unmapped_dsos = set(df_vnb["DSO"].dropna().unique()) - set(DSO_SHORT_NAMES.keys())
if unmapped_dsos:
    raise ValueError(f"Unmapped DSO names found in Zuordnung_PLZ-VNB.xlsx: {unmapped_dsos}")

plz_to_dso_short = (
    df_vnb.assign(PLZ=pd.to_numeric(df_vnb["PLZ"], errors="coerce").astype("Int64"), dso_short=df_vnb["DSO"].map(DSO_SHORT_NAMES))
    .dropna(subset=["PLZ"])
    .drop_duplicates(subset=["PLZ"], keep="first")
    .set_index("PLZ")["dso_short"]
)

# Look up DSO using the pylovo PLZ; fall back to 'nVNB' for border PLZs absent from the mapping file
df_grids["dso_short"] = df_grids["plz_pylovo"].astype("Int64").map(plz_to_dso_short).fillna("nVNB")

# Sort deterministically so 5-digit grid_index (00000..max) stays reproducible across reruns
df_grids = df_grids.sort_values(by=["plz_pylovo", "kcid", "bcid"]).reset_index(drop=True)
df_grids["grid_index"] = df_grids.index.map(lambda idx: f"{idx:05d}")
print(df_grids["dso_short"].value_counts())
df_grids.head()

dso_short
ByWrk    38871
nVNB     24469
LEW       7011
Nrgie     6446
SWM       4230
AllgN     1651
SWA        880
RegB       858
MFrak      841
SWU        356
Name: count, dtype: int64


,plz_pylovo,kcid,bcid,x,y,pop_density_1km2_cell,x_census,y_census,lon,lat,census_lon,census_lat,plz,gemeindeschlüssel,name,regio7,dso_short,grid_index
0,63739,1,1,4.260828e+06,2.985693e+06,4617.0,4260500.0,2985500.0,9.161145,49.980566,9.156605,49.978797,63739,09661000,Aschaffenburg,75,nVNB,00000
1,63739,1,2,4.260925e+06,2.985840e+06,4617.0,4260500.0,2985500.0,9.162475,49.981900,9.156605,49.978797,63739,09661000,Aschaffenburg,75,nVNB,00001
2,63739,1,3,4.261032e+06,2.985056e+06,1164.0,4261500.0,2985500.0,9.164094,49.974858,9.170544,49.978899,63739,09661000,Aschaffenburg,75,nVNB,00002
3,63739,1,4,4.260984e+06,2.985950e+06,4617.0,4260500.0,2985500.0,9.163274,49.982890,9.156605,49.978797,63739,09661000,Aschaffenburg,75,nVNB,00003
4,63739,1,5,4.260892e+06,2.984628e+06,8133.0,4260500.0,2984500.0,9.162206,49.970994,9.156763,49.969806,63739,09661000,Aschaffenburg,75,nVNB,00004


### Read out pylovo grid data and export `.h5` files to DSS (`config.STORAGE_DIR`)

In [13]:
### Create database object to read grid data from pylovo database
DB = dbrd.DataBase()
DB.show_contents()

Available tables:
['schema_migrations', 'version', 'equipment_data', 'postcode', 'postcode_result', 'grid_result', 'lines_result', 'lines_result_helper', 'split_points', 'lines_result_cache', 'consumer_categories', 'mv_graph', 'mv_graph_vertex', 'mv_graph_edge', 'mv_graph_settlement', 'mv_variant', 'mv_substation', 'mv_grid', 'mv_node', 'mv_line', 'mv_ring', 'sample_set', 'municipal_register', 'clustering_parameters', 'classification_version', 'pandapower_bus', 'pandapower_line', 'egon_hvmv_substation', 'pandapower_load', 'load_edit', 'pandapower_trafo', 'ways_result', 'transformers', 'transformer_positions', 'transformer_classified', 'ags_log', 'building_exclusion', 'plz_parameters', 'transformer_positions_with_grid', 'lines_result_view', 'mv_variant_stats', 'buildings_result', 'mv_customers', 'buildings_result_with_grid']


In [14]:
def retrieve_pylovo_grid(df_region_specs, precomputed_weather=None):
    ''' :: df_region_specs must at least include columns:
        - grid_index, dso_short
        - plz_pylovo
        - kcid, bcid
        - plz
        - regio7
        - lat, lon
    '''
    pexp.export_single_grid(DB, df_region_specs, precomputed_weather, config.STORAGE_DIR)

In [16]:
# Safety check: if writing to hpc_mount, ensure the remote DSS folder is actually mounted
# so files are never written accidentally to the local disk.
if os.path.basename(os.path.normpath(config.STORAGE_DIR)) == "hpc_mount" and not os.path.ismount(config.STORAGE_DIR):
    raise RuntimeError(
        f"'{config.STORAGE_DIR}' is not mounted! Mount the DSS target via sshfs before running this cell."
    )

# List existing .h5 files once (much faster over sshfs than per-row os.path.exists calls)
existing_files = set(os.listdir(config.STORAGE_DIR)) if os.path.isdir(config.STORAGE_DIR) else set()

def grid_exists(row):
    cell_id = f"N{int(row['y'])}E{int(row['x'])}"
    filename = svgrd.build_grid_filename(
        row["grid_index"],
        row["dso_short"],
        cell_id,
        row["plz_pylovo"],
        row["kcid"],
        row["bcid"],
    )
    return filename in existing_files

df_remaining_grids = df_grids[~df_grids.apply(grid_exists, axis=1)]
print(f"Grids left to process: {len(df_remaining_grids)} / {len(df_grids)}")

# Parallel export across isolated worker processes (each stages .h5 locally in /tmp before 1 atomic copy to hpc_mount)
MAX_WORKERS = 8
export_summary = pexp.run_parallel_export(
    df_remaining_grids,
    storage_dir=config.STORAGE_DIR,
    max_workers=MAX_WORKERS,
    progress_every_cells=100,
)

Grids left to process: 1 / 85613
Target storage directory: /home/pedro/Linux-AntigravityProjects/SurroGrid-linux/hpc_mount
Starting parallel export with 8 workers: 1 grids across 1 unique 1 km Zensus cells
Progress: 1/1 grids | 1/1 weather cells | 0.24 grids/s | elapsed 0.1 min | ETA ~0.0 h | errors=0
